In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LogicLayer(nn.Module):
    def __init__(self, num_tokens, gate_types, gates_per_type):
        super().__init__()
        self.num_tokens = num_tokens
        self.gate_types = gate_types
        self.gates_per_type = gates_per_type
        
        # Learnable routing weights: one per token per gate
        # Shape: [num_tokens, num_gate_types, gates_per_type]
        self.routing_weights = nn.Parameter(
            torch.randn(num_tokens, len(gate_types), gates_per_type)
        )
        
    def forward(self, tokens, train=True):
        # tokens: [num_tokens, hidden_dim]
        token_gate_outputs = torch.zeros_like(tokens)  # or other combination scheme
        
        for i, gate_type in enumerate(self.gate_types):
            # Softmax along gates_per_type dimension
            alpha = F.softmax(self.routing_weights[:, i, :], dim=-1)  # [num_tokens, gates_per_type]
            
            # Gather candidate inputs for each gate (here we just use token itself or neighbors)
            # Simplifying: assume each token has one input per gate for now
            gate_inputs = tokens.unsqueeze(1).expand(-1, self.gates_per_type, -1)  # [num_tokens, gates_per_type, hidden_dim]
            
            # Weighted sum by routing weights
            weighted_input = (alpha.unsqueeze(-1) * gate_inputs).sum(dim=1)  # [num_tokens, hidden_dim]
            
            # Apply logic function (e.g., AND = multiplication)
            # Use sigmoid for differentiable approximation
            gate_output = torch.sigmoid(weighted_input) if train else (weighted_input > 0).float()
            
            token_gate_outputs += gate_output  # combine outputs (could be sum, concat, etc.)
        
        return token_gate_outputs